# 🏥 Phân Đoạn Vùng Tổn Thương X-Quang — So Sánh 3 Mô Hình

**Notebook này chứa đầy đủ pipeline cho 3 mô hình:**
| # | Mô hình | Vai trò |
|---|---------|--------|
| 1 | **U-Net 2D** | Baseline gốc |
| 2 | **Attention U-Net 2D** | Baseline cải tiến |
| 3 | **PGA-UNet 2D** | Đề xuất (Prompt-Guided Attention) |

---
**Hướng dẫn sử dụng:**
- Chạy **Cell 1 (Setup)** trước — bắt buộc cho tất cả.
- Sau đó chạy phần nào cần: Section 2, 3, hoặc 4.
- Section 5 (Demo App) chạy sau khi đã có checkpoint PGA-UNet.

In [ ]:
# =========================================================
# CELL 1 — SETUP CHUNG (chạy 1 lần duy nhất)
# =========================================================
%cd /content

# Kiểm tra GPU
import torch
print('=' * 50)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
print('=' * 50)

# Clone repo (1 branch duy nhất chứa cả 3 model)
# ⚠️ Đổi branch name cho đúng sau khi push code lên GitHub
!git clone -b main https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git

# Download dataset (cho UNet + AttUNet baseline — có folder masks/)
!gdown --id 1Yg97cO5W2pYKV30wJQCl24xOEKQCwunI   # dataset_BTXRD.zip
!unzip -q dataset_BTXRD.zip
!mv dataset_BTXRD Prompt-Guided-XRay-Segmentation/

%cd Prompt-Guided-XRay-Segmentation

# Cài thư viện
!pip install -q tqdm opencv-python matplotlib scikit-image gdown scipy

print('\n✅ SETUP DONE!')

---
## 📦 Section 2 — U-Net 2D (Baseline Gốc)

- Dataset: mask PNG, `IMG_SIZE=256`
- Script: `train_unet.py`
- Checkpoint lưu tại: `checkpoints/unet_best.pth`

In [ ]:
# =========================================================
# TRAIN — U-Net 2D
# =========================================================
!python train_unet.py

In [ ]:
# =========================================================
# TEST — U-Net 2D  (Dice / IoU / HD95 / CBL)
# =========================================================
import os, cv2, torch, numpy as np, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from scipy.ndimage import binary_erosion, distance_transform_edt

from dataset_simple import BTXRD_Dataset
from models.networks.unet_2D import unet_2D

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = 'checkpoints/unet_best.pth'
IMG_SIZE   = 256
SHOW_INDEX = [0, 1, 2]

# ── Helpers ──────────────────────────────────────────────
def extract_lcc(m):
    if m.sum() == 0: return m
    nu, la, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), 8)
    if nu <= 1: return m
    return (la == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])).astype(np.float32)

def calc_hd95(p, g):
    p, g = p.astype(bool), g.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or  not g.any(): return float(IMG_SIZE)
    d1 = distance_transform_edt(~g.astype(bool) if g.any() else ~p)[p ^ binary_erosion(p)]
    d2 = distance_transform_edt(~p.astype(bool) if p.any() else ~g)[g ^ binary_erosion(g)]
    if not len(d1) or not len(d2): return float(IMG_SIZE)
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_hd95(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or  not gt.any(): return float(IMG_SIZE)
    pe = pred ^ binary_erosion(pred)
    ge = gt   ^ binary_erosion(gt)
    d1 = distance_transform_edt(~ge)[pe]
    d2 = distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return float(IMG_SIZE)
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_cbl(pm, gm):
    if gm.sum() == 0: return None
    ys, xs = np.where(gm)
    gt_diag = np.sqrt((ys.max()-ys.min())**2 + (xs.max()-xs.min())**2) + 1e-6
    if pm.sum() == 0: return 0.0
    yp, xp = np.where(pm)
    return float(np.clip(1.0 - np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/gt_diag, 0, 1))

def centroid(m):
    if m.sum() == 0: return None, None
    ys, xs = np.where(m); return float(xs.mean()), float(ys.mean())

# ── Load ─────────────────────────────────────────────────
model = unet_2D(in_channels=1, n_classes=1).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()

test_ds     = BTXRD_Dataset('dataset_BTXRD/test/images', 'dataset_BTXRD/test/masks', IMG_SIZE, is_train=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

# ── Evaluate ─────────────────────────────────────────────
all_dice, all_iou, all_pre, all_rec, all_hd95, all_cbl = [], [], [], [], [], []
smooth = 1e-5

with torch.no_grad():
    for idx, (imgs, masks) in enumerate(test_loader):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = (torch.sigmoid(model(imgs)) > 0.5).float()

        img_np = (imgs[0,0].cpu().numpy() + 1) / 2.0
        gm = masks[0,0].cpu().numpy()
        pm = extract_lcc(preds[0,0].cpu().numpy())

        tp = (pm*gm).sum(); fp = (pm*(1-gm)).sum(); fn = ((1-pm)*gm).sum()
        dice = (2*tp+smooth)/(2*tp+fp+fn+smooth)
        iou  = (tp+smooth)/(tp+fp+fn+smooth)
        pre  = (tp+smooth)/(tp+fp+smooth)
        rec  = (tp+smooth)/(tp+fn+smooth)
        hd   = calc_hd95(pm.astype(bool), gm.astype(bool))
        cbl  = calc_cbl(pm.astype(bool), gm.astype(bool))

        all_dice.append(dice); all_iou.append(iou)
        all_pre.append(pre);   all_rec.append(rec)
        all_hd95.append(hd)
        if cbl is not None: all_cbl.append(cbl)

        if idx in SHOW_INDEX:
            cx_gt, cy_gt = centroid(gm); cx_p, cy_p = centroid(pm)
            fig, axes = plt.subplots(1, 4, figsize=(20, 5))
            fig.suptitle(f'U-Net 2D | Sample {idx}', fontsize=14, fontweight='bold')
            axes[0].imshow(img_np, cmap='gray'); axes[0].set_title('Ảnh gốc')
            axes[1].imshow(img_np, cmap='gray')
            green = np.zeros((*gm.shape, 4), dtype=np.float32); green[gm==1]=[0,1,0,0.3]
            axes[1].imshow(green)
            if gm.max()>0: axes[1].contour(gm,[0.5],colors='lime',linewidths=1.5)
            if cx_gt: axes[1].plot(cx_gt,cy_gt,'o',color='lime',ms=8,markeredgecolor='black')
            axes[1].set_title('Ground Truth')
            axes[2].imshow(img_np, cmap='gray')
            red = np.zeros((*pm.shape, 4), dtype=np.float32); red[pm==1]=[1,0,0,0.3]
            axes[2].imshow(red)
            if pm.max()>0: axes[2].contour(pm,[0.5],colors='red',linewidths=1.5)
            if cx_p: axes[2].plot(cx_p,cy_p,'o',color='red',ms=8,markeredgecolor='white')
            axes[2].set_title('Dự đoán (U-Net)')
            axes[3].imshow(img_np, cmap='gray')
            if gm.max()>0: axes[3].contour(gm,[0.5],colors='lime',linewidths=2)
            if pm.max()>0: axes[3].contour(pm,[0.5],colors='red',linewidths=2,linestyles='--')
            if cx_gt: axes[3].plot(cx_gt,cy_gt,'o',color='lime',ms=8,markeredgecolor='black')
            if cx_p:  axes[3].plot(cx_p,cy_p,'o',color='red',ms=8,markeredgecolor='white')
            if cx_gt and cx_p: axes[3].plot([cx_gt,cx_p],[cy_gt,cy_p],'--',color='yellow',lw=1.5)
            axes[3].set_title(f'Dice:{dice:.3f} IoU:{iou:.3f}\nHD95:{hd:.1f}px CBL:{cbl:.3f}')
            for ax in axes: ax.axis('off')
            plt.tight_layout(); plt.show()

print('\n' + '='*60)
print('📊 TEST RESULTS — U-NET 2D')
print('='*60)
print(f'Dice ↑      : {np.mean(all_dice):.4f}')
print(f'IoU  ↑      : {np.mean(all_iou):.4f}')
print(f'Precision ↑ : {np.mean(all_pre):.4f}')
print(f'Recall ↑    : {np.mean(all_rec):.4f}')
print(f'HD95 ↓ (px) : {np.mean(all_hd95):.2f}')
print(f'CBL ↑       : {np.mean(all_cbl):.4f}')
print(f'Samples     : {len(all_dice)}')
print('='*60)

---
## 📦 Section 3 — Attention U-Net 2D (Baseline Cải Tiến)

- Dataset: mask PNG, `IMG_SIZE=256`
- Script: `train_attunet.py`
- Checkpoint lưu tại: `checkpoints/attunet_best.pth`

In [ ]:
# =========================================================
# TRAIN — Attention U-Net 2D
# =========================================================
!python train_attunet.py

In [ ]:
# =========================================================
# TEST — Attention U-Net 2D
# =========================================================
import os, cv2, torch, numpy as np, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from scipy.ndimage import binary_erosion, distance_transform_edt

from dataset_simple import BTXRD_Dataset
from models.networks.attention_unet_2D import Attention_UNet_2D

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = 'checkpoints/attunet_best.pth'
IMG_SIZE   = 256
SHOW_INDEX = [0, 5, 10]

# ── Helpers (dùng chung với UNet) ────────────────────────
def extract_lcc(m):
    if m.sum() == 0: return m
    nu, la, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), 8)
    if nu <= 1: return m
    return (la == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])).astype(np.float32)

def calc_hd95(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or  not gt.any(): return float(IMG_SIZE)
    pe = pred ^ binary_erosion(pred)
    ge = gt   ^ binary_erosion(gt)
    d1 = distance_transform_edt(~ge)[pe]
    d2 = distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return float(IMG_SIZE)
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_cbl(pm, gm):
    if gm.sum() == 0: return None
    ys, xs = np.where(gm)
    gt_diag = np.sqrt((ys.max()-ys.min())**2 + (xs.max()-xs.min())**2) + 1e-6
    if pm.sum() == 0: return 0.0
    yp, xp = np.where(pm)
    return float(np.clip(1.0 - np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/gt_diag, 0, 1))

def centroid(m):
    if m.sum() == 0: return None, None
    ys, xs = np.where(m); return float(xs.mean()), float(ys.mean())

# ── Load ─────────────────────────────────────────────────
model = Attention_UNet_2D(in_channels=1, n_classes=1).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()

test_ds     = BTXRD_Dataset('dataset_BTXRD/test/images', 'dataset_BTXRD/test/masks', IMG_SIZE, is_train=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

# ── Evaluate ─────────────────────────────────────────────
all_dice, all_iou, all_pre, all_rec, all_hd95, all_cbl = [], [], [], [], [], []
smooth = 1e-5

with torch.no_grad():
    for idx, (imgs, masks) in enumerate(test_loader):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = (torch.sigmoid(model(imgs)) > 0.5).float()

        img_np = (imgs[0,0].cpu().numpy() + 1) / 2.0
        gm = masks[0,0].cpu().numpy()
        pm = extract_lcc(preds[0,0].cpu().numpy())

        tp = (pm*gm).sum(); fp = (pm*(1-gm)).sum(); fn = ((1-pm)*gm).sum()
        dice = (2*tp+smooth)/(2*tp+fp+fn+smooth)
        iou  = (tp+smooth)/(tp+fp+fn+smooth)
        pre  = (tp+smooth)/(tp+fp+smooth)
        rec  = (tp+smooth)/(tp+fn+smooth)
        hd   = calc_hd95(pm.astype(bool), gm.astype(bool))
        cbl  = calc_cbl(pm.astype(bool), gm.astype(bool))

        all_dice.append(dice); all_iou.append(iou)
        all_pre.append(pre);   all_rec.append(rec)
        all_hd95.append(hd)
        if cbl is not None: all_cbl.append(cbl)

        if idx in SHOW_INDEX:
            cx_gt, cy_gt = centroid(gm); cx_p, cy_p = centroid(pm)
            fig, axes = plt.subplots(1, 4, figsize=(20, 5))
            fig.suptitle(f'Attention U-Net 2D | Sample {idx}', fontsize=14, fontweight='bold')
            axes[0].imshow(img_np, cmap='gray'); axes[0].set_title('Ảnh gốc')
            axes[1].imshow(img_np, cmap='gray')
            green = np.zeros((*gm.shape, 4), dtype=np.float32); green[gm==1]=[0,1,0,0.35]
            axes[1].imshow(green)
            if gm.max()>0: axes[1].contour(gm,[0.5],colors='lime',linewidths=1.5)
            if cx_gt: axes[1].plot(cx_gt,cy_gt,'o',color='lime',ms=8,markeredgecolor='black')
            axes[1].set_title('Ground Truth')
            axes[2].imshow(img_np, cmap='gray')
            red = np.zeros((*pm.shape, 4), dtype=np.float32); red[pm==1]=[1,0,0,0.35]
            axes[2].imshow(red)
            if pm.max()>0: axes[2].contour(pm,[0.5],colors='red',linewidths=1.5)
            if cx_p: axes[2].plot(cx_p,cy_p,'o',color='red',ms=8,markeredgecolor='white')
            axes[2].set_title('Dự đoán (Att-UNet)')
            axes[3].imshow(img_np, cmap='gray')
            if gm.max()>0: axes[3].contour(gm,[0.5],colors='lime',linewidths=2)
            if pm.max()>0: axes[3].contour(pm,[0.5],colors='red',linewidths=2,linestyles='--')
            if cx_gt: axes[3].plot(cx_gt,cy_gt,'o',color='lime',ms=8,markeredgecolor='black')
            if cx_p:  axes[3].plot(cx_p,cy_p,'o',color='red',ms=8,markeredgecolor='white')
            if cx_gt and cx_p: axes[3].plot([cx_gt,cx_p],[cy_gt,cy_p],'--',color='yellow',lw=1.5)
            axes[3].set_title(f'Dice:{dice:.3f} IoU:{iou:.3f}\nHD95:{hd:.1f}px CBL:{cbl:.3f}')
            for ax in axes: ax.axis('off')
            plt.tight_layout(); plt.show()

print('\n' + '='*60)
print('📊 TEST RESULTS — ATTENTION U-NET 2D')
print('='*60)
print(f'Dice ↑      : {np.mean(all_dice):.4f}')
print(f'IoU  ↑      : {np.mean(all_iou):.4f}')
print(f'Precision ↑ : {np.mean(all_pre):.4f}')
print(f'Recall ↑    : {np.mean(all_rec):.4f}')
print(f'HD95 ↓ (px) : {np.mean(all_hd95):.2f}')
print(f'CBL ↑       : {np.mean(all_cbl):.4f}')
print(f'Samples     : {len(all_dice)}')
print('='*60)

---
## 📦 Section 4 — PGA-UNet 2D (Đề Xuất)

- Dataset: **JSON polygon** (annotations/), `IMG_SIZE=512`
- Script: `train_pga.py` — cấu hình `EXPERIMENT='B'` (mixed 70-30)
- Checkpoints lưu tại: `checkpoints/pga_unet_expB_best.pth`

**Hai thí nghiệm:**
- **Exp A** (`EXPERIMENT='A'`): prompt zoom-out only → huấn luyện sạch
- **Exp B** (`EXPERIMENT='B'`): prompt mixed 70% zoom-out + 30% shift → robust hơn

In [ ]:
# =========================================================
# (Tuỳ chọn) Tải checkpoint PGA đã train sẵn từ Google Drive
# =========================================================
import gdown, os
os.makedirs('checkpoints', exist_ok=True)

# Checkpoint Exp B (PGA-UNet)
gdown.download('https://drive.google.com/uc?id=1Mv-rUPI7KGmYemd27hmKbJQRHc4ZKB9z',
               'checkpoints/pga_unet_expB_best.pth', quiet=False)

# Checkpoint MobileNetV4 (Classifier sàng lọc — dùng cho App)
gdown.download('https://drive.google.com/uc?id=1N-Xq4l3qOcwDhRCu5g-wFQindT8ubOgx',
               'checkpoints/best_mobilenetv4.pth', quiet=False)

print('✅ Checkpoint tải xong!')

In [ ]:
# =========================================================
# TRAIN — PGA-UNet 2D
# (Bỏ qua nếu đã tải checkpoint ở trên)
# Đổi EXPERIMENT trong train_pga.py: 'A' hoặc 'B'
# =========================================================
!python train_pga.py

In [ ]:
# =========================================================
# TEST — PGA-UNet 2D (Đánh giá đầy đủ với IPR + GradCAM)
# =========================================================
import os, cv2, torch, numpy as np, matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader
from scipy.ndimage import binary_erosion, distance_transform_edt
from tqdm import tqdm

from dataset import BTXRD_Dataset
from models.networks.prompt_unet_2D import PGA_UNet

# ── Cấu hình ─────────────────────────────────────────────
DEVICE             = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH         = 'checkpoints/pga_unet_expB_best.pth'
IMG_SIZE           = 512
USE_ENCODER_PROMPT = True
NUM_IPR_STEPS      = 5        # Số vòng lặp IPR (Iterative Prompt Refinement)
NUM_VISUALIZE      = 5        # Số ảnh hiển thị mẫu
DARK_PIXEL_THRESHOLD = -0.80  # Ngưỡng phát hiện prompt chứa vùng đen
DARK_RATIO_LIMIT     = 0.70   # Tỷ lệ vùng đen tối đa cho phép trong prompt

# ── Helpers ──────────────────────────────────────────────
def extract_lcc(m):
    if m.sum() == 0: return m
    nu, la, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), 8)
    if nu <= 1: return m
    return (la == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])).astype(np.float32)

def calc_metrics(pm, gm):
    pm, gm = (pm > 0.5).astype(np.float32), (gm > 0.5).astype(np.float32)
    tp = (pm * gm).sum(); fp = (pm * (1-gm)).sum(); fn = ((1-pm) * gm).sum()
    dice = (2*tp + 1e-5) / (2*tp + fp + fn + 1e-5)
    iou  = (tp + 1e-5) / (tp + fp + fn + 1e-5)
    return dice, iou

def calc_hd95(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or  not gt.any(): return float(IMG_SIZE)
    pe = pred ^ binary_erosion(pred); ge = gt ^ binary_erosion(gt)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return float(IMG_SIZE)
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_cbl(pm, gm):
    if gm.sum() == 0 or pm.sum() == 0: return 0.0
    ys, xs = np.where(gm); yp, xp = np.where(pm)
    gt_diag = np.sqrt((ys.max()-ys.min())**2 + (xs.max()-xs.min())**2) + 1e-6
    return float(np.clip(1.0 - np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/gt_diag, 0, 1))

def centroid(m):
    if m.sum() == 0: return None, None
    ys, xs = np.where(m > 0.5); return float(xs.mean()), float(ys.mean())

def compute_gradcam(model, img_tensor):
    gradients, activations = [], []
    def fwd_hook(module, inp, out):
        activations.append(out)
        out.register_hook(lambda g: gradients.append(g))
    hook = model.center.register_forward_hook(fwd_hook)
    model.eval()
    img_t = img_tensor.clone().detach().to(DEVICE)
    zero_p = torch.zeros(1, 1, IMG_SIZE, IMG_SIZE, device=DEVICE)
    out = model(img_t, zero_p); model.zero_grad(); out.sum().backward()
    hook.remove()
    if not gradients: return None
    w = gradients[0].mean(dim=(2,3), keepdim=True)
    cam = F.relu((w * activations[0]).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
    cam = cam[0,0].detach().cpu().numpy()
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

def is_dark_prompt(prompt_tensor):
    """Phát hiện prompt chứa quá nhiều vùng đen (prompt giả/tấn công)."""
    p = prompt_tensor[0, 0].cpu().numpy()
    return (p < DARK_PIXEL_THRESHOLD).mean() > DARK_RATIO_LIMIT

def ipr_refine(model, img_tensor, init_prompt):
    """Iterative Prompt Refinement: dùng prediction làm prompt vòng tiếp theo."""
    prompt = init_prompt.clone()
    for _ in range(NUM_IPR_STEPS):
        if is_dark_prompt(prompt):
            prompt = torch.zeros_like(prompt)  # phòng vệ
        with torch.no_grad():
            out  = model(img_tensor.to(DEVICE), prompt.to(DEVICE))
            prob = torch.sigmoid(out)
            pred = (prob > 0.5).float()
        # Tạo prompt mới từ prediction (centroid-based heatmap)
        pred_np = pred[0, 0].cpu().numpy()
        cx, cy  = centroid(pred_np)
        if cx is None: break
        new_p = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)
        xs, ys = np.where(pred_np > 0.5)
        if len(xs):
            r = max(xs.max()-xs.min(), ys.max()-ys.min()) * 0.3
            cv2.circle(new_p, (int(cy), int(cx)), max(1, int(r)), 1.0, -1)
            new_p = cv2.GaussianBlur(new_p, (31, 31), 0)
        prompt = torch.from_numpy(new_p).unsqueeze(0).unsqueeze(0)
    return pred, prob, prompt

# ── Load model ────────────────────────────────────────────
model = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=USE_ENCODER_PROMPT).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()

# ── Dataset (test với zoom_out cho fair comparison) ───────
test_ds = BTXRD_Dataset(
    image_dir='dataset_BTXRD/test/images',
    json_dir='dataset_BTXRD/test/annotations',
    img_size=IMG_SIZE, is_train=False,
    prompt_mode='zoom_out'
)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

# ── Evaluate ─────────────────────────────────────────────
all_dice, all_iou, all_hd95, all_cbl = [], [], [], []
vis_count = 0

for idx, (imgs, masks, prompts) in enumerate(tqdm(test_loader, desc='Testing PGA-UNet')):
    imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

    pred, prob, _ = ipr_refine(model, imgs, prompts)

    gm = masks[0, 0].cpu().numpy()
    pm = extract_lcc(pred[0, 0].cpu().numpy())

    dice, iou = calc_metrics(pm, gm)
    hd   = calc_hd95(pm.astype(bool), gm.astype(bool))
    cbl  = calc_cbl(pm.astype(bool), gm.astype(bool))

    all_dice.append(dice); all_iou.append(iou)
    all_hd95.append(hd);   all_cbl.append(cbl)

    if vis_count < NUM_VISUALIZE:
        img_np = (imgs[0,0].cpu().numpy() + 1) / 2.0
        cx_gt, cy_gt = centroid(gm); cx_p, cy_p = centroid(pm)
        cam = compute_gradcam(model, imgs)

        fig, axes = plt.subplots(1, 5, figsize=(25, 5))
        fig.suptitle(f'PGA-UNet 2D | Sample {idx}', fontsize=14, fontweight='bold')
        axes[0].imshow(img_np, cmap='gray'); axes[0].set_title('Ảnh gốc')
        prompt_np = prompts[0, 0].cpu().numpy()
        axes[1].imshow(img_np, cmap='gray')
        axes[1].imshow(prompt_np, cmap='hot', alpha=0.5); axes[1].set_title('Prompt')
        axes[2].imshow(img_np, cmap='gray')
        green = np.zeros((*gm.shape, 4), dtype=np.float32); green[gm==1]=[0,1,0,0.35]
        axes[2].imshow(green)
        if gm.max()>0: axes[2].contour(gm,[0.5],colors='lime',linewidths=1.5)
        if cx_gt: axes[2].plot(cx_gt,cy_gt,'o',color='lime',ms=8,markeredgecolor='black')
        axes[2].set_title('Ground Truth')
        axes[3].imshow(img_np, cmap='gray')
        red = np.zeros((*pm.shape, 4), dtype=np.float32); red[pm==1]=[1,0,0,0.35]
        axes[3].imshow(red)
        if pm.max()>0: axes[3].contour(pm,[0.5],colors='red',linewidths=1.5)
        if cx_p: axes[3].plot(cx_p,cy_p,'o',color='red',ms=8,markeredgecolor='white')
        axes[3].set_title('Dự đoán (PGA-UNet)')
        if cam is not None:
            axes[4].imshow(img_np, cmap='gray')
            axes[4].imshow(cam, cmap='jet', alpha=0.5)
        axes[4].set_title(f'GradCAM\nDice:{dice:.3f} IoU:{iou:.3f}\nHD95:{hd:.1f}px CBL:{cbl:.3f}')
        for ax in axes: ax.axis('off')
        plt.tight_layout(); plt.show()
        vis_count += 1

print('\n' + '='*60)
print('📊 TEST RESULTS — PGA-UNET 2D (Exp B, zoom_out eval)')
print('='*60)
print(f'Dice ↑      : {np.mean(all_dice):.4f}')
print(f'IoU  ↑      : {np.mean(all_iou):.4f}')
print(f'HD95 ↓ (px) : {np.mean(all_hd95):.2f}')
print(f'CBL ↑       : {np.mean(all_cbl):.4f}')
print(f'Samples     : {len(all_dice)}')
print('='*60)

In [ ]:
# =========================================================
# DOWNLOAD Checkpoint + Log về máy
# =========================================================
!zip -r checkpoints_pga.zip checkpoints logs
from google.colab import files
files.download('checkpoints_pga.zip')

---
## 📦 Section 5 — Demo App (Gradio)

Chạy **sau khi** Section 4 đã có checkpoint PGA-UNet.

In [ ]:
# =========================================================
# Cài thêm thư viện cho App
# =========================================================
!pip install -q gradio openpyxl torchmetrics timm
!pip install --upgrade huggingface_hub gradio

In [ ]:
# =========================================================
# CHẠY GRADIO APP
# =========================================================
%run app.py